In [18]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

motion_list  = ['Flexion']
weight = 802
motion_folder = motion_list[0]
motion_name = motion_list[0]
data_struct = sc.io.loadmat('../data_model.mat')
OS_struct = sc.io.loadmat('../Motions/'+motion_folder+'/OS_model.mat')

act_w = 1
vel_w = 0.1

MM,FO,q,w,u0,fr,frstar,kinematical,xdot,holonomic,first_elips_scale,elips_trans = eq.create_eoms_u0state(data_struct,OS_struct,derive = 'numeric',gen_matlab_functions = 0)
TE,activations,TE_conoid = eq.polynomials_quat(model_struct = OS_struct,q = q,derive = 'numeric',model_params_struct = data_struct)

0.015


In [37]:
reload(eq)
reload(tr)
clav_pos = [0.2,0.3,0.4,0.5,0.6,0.7]
weights = [250,400,600,800]

for iweight in range(len(weights)):
    for ipos in range(len(clav_pos)):
        weight = int(weights[iweight] + clav_pos[ipos]*10)
        print(weight)
        struct_name = 'res_quat_'+motion_list[0]+'_'+str(weight)
        traj_w = weight
        excitations = []
        act_ode = []
        # print(activations)
        for i in range(len(activations)):
            excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
            act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i]))

        sp_act_ode = sp.Matrix(act_ode)
        eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+TE+sp.Matrix(TE_conoid)).col_join(holonomic).col_join(sp_act_ode)
        reload(eq)
        num_nodes = 101
        file = '../Motions/' + motion_folder + '/' + motion_name
        traj_original, interval_value, time = tr.exp_trajectory_quat(file,num_nodes)
        traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos[ipos])

        state_symbols = tuple(q+w+u0+activations)
        num_states = len(state_symbols)
        num_q = len(q)
        num_u = len(w+u0)
        specified_symbols = tuple(excitations)
        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        objective_traj,objective_traj_jac = eq.custom_objective_quat(num_q,interval_value,clav_pos[ipos])
        obj_min_diff,obj_min_diff_jac = eq.objective_activation_diff(num_nodes, interval_value)
        w_diff_vel = 2
        w_diff_act = 2
        w_diff_exc = 0.1
        node1 = 0
        node2 = num_nodes//2
        node3 = num_nodes-1

        def obj(free):
            # min_traj = traj_w * interval_value * np.sum((traj_original.flatten() - free[:10*num_nodes])**2)
            min_traj = traj_w * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj))
            # min_traj = traj_w * (objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node1],traj[:,node1]) + objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node2],traj[:,node2]) + objective_traj(np.array(np.split(free[:10*num_nodes],10))[:,node3],traj[:,node3]))

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
            min_torque = act_w * interval_value * np.sum(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes]**2)
            min_act_dif = w_diff_act * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))))
            min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))))

            # min_act_dif = obj_act_dif(np.ones((101,5)))
            return (min_traj + min_torque + min_vel_dif + min_act_dif + min_exc_dif).item()

        def obj_grad(free):
            grad = np.zeros_like(free)
            # grad[:10*num_nodes] = traj_w * 2.0 * interval_value * (free[:10*num_nodes] - traj_original.flatten())
            grad[:num_q*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj))

            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] = act_w * 2.0 * interval_value * free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] + w_diff_act * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
            grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] = w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))
            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] = w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))

            # ## reach ##
            # first_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node1],traj[:,node1])))
            # second_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node2],traj[:,node2])))
            # third_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:10*num_nodes],10))[:,node3],traj[:,node3])))
            # # print(np.shape(first_grad_vals))
            # grad_traj = np.zeros((num_nodes,10))
            # grad_traj[node1,:] = first_grad_vals
            # grad_traj[node2,:] = second_grad_vals
            # grad_traj[node3,:] = third_grad_vals
            # grad[:10*num_nodes] = traj_w * np.concatenate(grad_traj.T)
            # ## end reach ##

            return grad
        instance_constraints = []
        # for i in range(13):
        instance_constraints.append(state_symbols[-1].func(0.0)-0) 
            
        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
        bndrs.update(bndrs_exc)
        SCq_bndr = {q[0] : (0.6, 1)}
        bndrs.update(SCq_bndr)
        ACq_bndr = {q[4] : (0.6, 1)}
        bndrs.update(ACq_bndr)
        GHq_bndr = {q[8] : (0.4, 1)}
        bndrs.update(GHq_bndr)


        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint')


        time_to_create = tm.time() - start
        print(time_to_create)



        prob.add_option('max_iter',5000)
        prob.add_option('limited_memory_max_history', 40)
        initial_guess = np.zeros(prob.num_free)
        initial_guess[:13*num_nodes] = traj_original.flatten()
        time_2_solve_start = tm.time()
        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)

        reload(tr)
        file_name = '../Motions/'+motion_folder+'/' + struct_name + '.mat'
        tr.sol2struct(solution,activations,num_q,num_u,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,True)

        file_name_mot = '../Motions/'+motion_folder+'/' + struct_name + '.mot'
        tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot)

252
1917.2074327468872
This is Ipopt version 3.14.16, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:  8568001
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    28280
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    25957
                     variables with only upper bounds:        0
Total number of equality constraints.................:    15301
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  5.0025200e-02 4.86e+02 1.61e-03   0.0 0.00e+00  

/home/zoufaond/Shoulder_modelling/Python/trajectory_lib.py:277: RuntimeWarning: invalid value encountered in arccos
  z = np.arccos(rotm[1,1])


2031.2262103557587
This is Ipopt version 3.14.16, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:  8568001
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    28280
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    25957
                     variables with only upper bounds:        0
Total number of equality constraints.................:    15301
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  5.0025200e-02 4.86e+02 1.61e-03   0.0 0.00e+00    - 

KeyboardInterrupt: 

In [25]:
# from opty import Problem, create_objective_function, parse_free
# import sympy as sp
# import numpy as np
# import scipy as sc
# import time as tm
# import pickle
# import sympy.physics.mechanics as me
# import sys
# sys.path.insert(0, "..")
# from importlib import reload
# import matplotlib.pyplot as plt
# import equations as eq
# reload (eq);
# import trajectory_lib as tr
# reload (tr);

# initPos = 'InitPosOptQuat'

# # motion_folder_list = ['Elevation','Abduction_rigged2','Abduction_rigged','Abduction','Steering']
# # motion_list = ['elevation.mat','abd_rigged.mat','abd_rigged.mat','abduction.mat','steering.mat']
# # weights_list = [100,150,200,250]

# # motion_folder_list = ['Flexion_noised','Scabduction_noised']
# # motion_list = ['flexion_simulation.mat','scabduction_GL.mat']
# motion_list  = ['Elevation_1']
# # weights_list = [100,150,200,250,300]
# weight = 305

# data_struct = sc.io.loadmat('../data_model.mat')
# MM,FO,q,w,u0,fr,frstar,kinematical,xdot,holonomic,first_elips_scale,elips_trans = eq.create_eoms_u0state(data_struct,derive = 'numeric',gen_matlab_functions = 0)


# motion_folder = motion_list[0]
# motion_name = motion_list[0]
# act_w = 1
# vel_w = 0.1
# model_struct = sc.io.loadmat('../Motions/'+motion_folder+'/OS_model.mat')

# TE,activations,TE_conoid = eq.polynomials_quat(model_struct = model_struct,q = q,derive = 'numeric',model_params_struct = data_struct)
# # q = eq.create_eoms_u0state(model_struct,data_struct,initPos,derive = 'numeric')


In [26]:

# struct_name = 'res_quat_'+motion_list[0]+'_'+str(weight)
# traj_w = weight
# # dict_vals,symlist, value_list = eq.create_parameters_dict(data_struct, initPos)
# x0 = data_struct['params'][initPos][0,0]['initCondQuat'].item()
# x0t = list(x0.T[0])
# eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+TE+sp.Matrix(TE_conoid)).col_join(holonomic)
# num_nodes = 101
# file = '../Motions/' + motion_folder + '/' + motion_name
# traj_original, interval_value, time = tr.exp_trajectory_quat(file,num_nodes)
# traj = tr.exp_trajectory_quat_myobj(traj_original)

# state_symbols = tuple(q+w+u0)
# num_states = len(state_symbols)
# specified_symbols = tuple(activations)
# num_inputs = len(specified_symbols)
# t = me.dynamicsymbols._t
# objective_traj,objective_traj_jac = eq.custom_objective_quat(len(q),interval_value)
# node1 = 0
# node2 = num_nodes//2
# node3 = num_nodes-1

# def obj(free):
#     # min_traj = traj_w * interval_value * np.sum((traj_original.flatten() - free[:13*num_nodes])**2)
#     # min_traj = traj_w * np.sum(objective_traj(np.split(free[:13*num_nodes],13),traj))

#     min_traj = traj_w * (objective_traj(np.array(np.split(free[:13*num_nodes],13))[:,node1],traj[:,node1]) + objective_traj(np.array(np.split(free[:13*num_nodes],13))[:,node2],traj[:,node2]) + objective_traj(np.array(np.split(free[:13*num_nodes],13))[:,node3],traj[:,node3]))

#     min_vel = vel_w * interval_value * np.sum((free[13*num_nodes:num_states*num_nodes])**2)
#     min_torque = act_w * interval_value * np.sum(free[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
#     return min_traj + min_torque + min_vel

# def obj_grad(free):
#     grad = np.zeros_like(free)
#     # grad[:13*num_nodes] = traj_w * 2.0 * interval_value * (free[:13*num_nodes] - traj_original.flatten())
#     # grad[:13*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:13*num_nodes],13),traj))
#     first_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:13*num_nodes],13))[:,node1],traj[:,node1])))
#     second_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:13*num_nodes],13))[:,node2],traj[:,node2])))
#     third_grad_vals = np.array((objective_traj_jac(np.array(np.split(free[:13*num_nodes],13))[:,node3],traj[:,node3])))
#     # print(np.shape(first_grad_vals))
#     grad_traj = np.zeros((num_nodes,13))
#     grad_traj[node1,:] = first_grad_vals
#     grad_traj[node2,:] = second_grad_vals
#     grad_traj[node3,:] = third_grad_vals
#     grad[:13*num_nodes] = traj_w * np.concatenate(grad_traj.T)

#     grad[13*num_nodes:num_states*num_nodes] = vel_w * 2 * interval_value * free[13*num_nodes:num_states*num_nodes]
#     grad[num_states*num_nodes:(num_states + num_inputs)*num_nodes] = act_w * 2.0 * interval_value * free[num_states*num_nodes:(num_states + num_inputs)*num_nodes]
#     return grad

# instance_constraints = []
# # for i in range(13):
# instance_constraints.append(state_symbols[-3].func(0.0)-0) 
    
# bounds1 = (0.0,1.0)
# bounds = (bounds1,)*len(activations)
# bndrs = dict(zip(activations,bounds))
# # bndrs.update({elips_trans[0]: (-0.1, 0.1),
# #               elips_trans[1]: (-0.25, -0.15),
# #               elips_trans[2]: (-0.05, 0.15)})
# # bndrs.update({first_elips_scale[0]: (1, 2),
# #               first_elips_scale[1]: (1, 2),
# #               first_elips_scale[2]: (1, 2)})

In [27]:

# start = tm.time()
# prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
#             num_nodes, interval_value,
#             known_parameter_map={},
#             instance_constraints=instance_constraints,
#             bounds=bndrs,
#             integration_method='midpoint')


# time_to_create = tm.time() - start
# print(time_to_create)



# prob.add_option('max_iter',2500)
# prob.add_option('limited_memory_max_history', 40)
# initial_guess = np.zeros(prob.num_free)
# initial_guess[:13*num_nodes] = traj_original.flatten()
# # initial_guess[-4] = 0.0621
# # initial_guess[-5] = -0.1521
# # initial_guess[-6] = 0.0
# # initial_guess[-1] = 1.1
# # initial_guess[-2] = 1.1
# # initial_guess[-3] = 1.1
# time_2_solve_start = tm.time()
# solution, info = prob.solve(initial_guess)
# time_2_solve = tm.time() - time_2_solve_start
# print(info['status_msg'])
# print(info['obj_val'])
# act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
# print('Objective activations: ', act_obj)
# reload (tr);
# # import matplotlib.pyplot as plt
# # tr.plot_results(solution,traj,time,num_nodes)
# # import matplotlib.pyplot as plt
# # fig, axes = plt.subplots(int(num_states+num_inputs), 1, sharex=True,
# #                          figsize=(6.4, 0.8*(num_states+num_inputs)),
# #                          layout='compressed')
# # prob.plot_trajectories(solution, axes=axes)
# # import matplotlib.pyplot as plt
# # fig, axs = plt.subplots(13)
# # for j in range(13):
# #     axs[j].plot(time,traj[j,:])
# #     axs[j].plot(time,solution[j*num_nodes:(j+1)*num_nodes])
# #     fig.set_figheight(10)

# # num_iter_sol = int(input('enter number of iterarions:'))
# file_name = '../Motions/'+motion_folder+'/' + struct_name + '.mat'
# tr.sol2struct(solution,activations,len(q),num_states,num_nodes,time,0,time_2_solve,file_name)

# file_name_mot = '../Motions/'+motion_folder+'/' + struct_name + '.mot'
# tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot)
# # print('elips_trans: ', solution[-6], solution[-5], solution[-4])
# # print('elips_Scale: ', solution[-1], solution[-2], solution[-3])

In [28]:
# from opty import Problem, create_objective_function, parse_free
# import sympy as sp
# import numpy as np
# import scipy as sc
# import time as tm
# import pickle
# import sympy.physics.mechanics as me
# import sys
# sys.path.insert(0, "..")
# from importlib import reload
# import matplotlib.pyplot as plt

# initPos = 'InitPosOptQuat'

# motion_folder_list = ['Abduction_rigged2','Elevation','Abduction_rigged','Abduction','Steering']
# motion_list = ['abd_rigged.mat','elevation.mat','abd_rigged.mat','abduction.mat','steering.mat']
# weights_list = [100,150,200,250,300]

# for i in range(len(motion_folder_list)):
#     for iweight in range(len(weights_list)):
#         motion_folder = motion_folder_list[i]
#         motion_name = motion_list[i]
#         struct_name = 'results_quat_QuatInit_'+str(weights_list[iweight])
#         act_w = 1
#         traj_w = weights_list[iweight]
#         vel_w = 0.1

#         model_struct = sc.io.loadmat('../Motions/'+motion_folder+'/OS_model.mat')
#         data_struct = sc.io.loadmat('../data_model.mat')
#         start = tm.time()
#         MM,FO,q,w,u0,fr,frstar,kinematical,xdot,holonomic = eq.create_eoms_u0state(model_struct,data_struct,initPos,derive = 'numeric')
#         TE,activations = eq.polynomials_quat(model_struct = model_struct,q = q,derive = 'numeric',model_params_struct = data_struct ,initCond_name = initPos)
#         # q = eq.create_eoms_u0state(model_struct,data_struct,initPos,derive = 'numeric')

#         time_to_create = tm.time() - start
#         print(time_to_create)
#         import equations as eq
#         reload (eq);
#         # dict_vals,symlist, value_list = eq.create_parameters_dict(data_struct, initPos)
#         x0 = data_struct['params'][initPos][0,0]['initCondQuat'].item()
#         x0t = list(x0.T[0])
#         eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+TE).col_join(holonomic)
#         import trajectory_lib as tr
#         reload (tr);
#         num_nodes = 101
#         file = '../Motions/' + motion_folder + '/' + motion_name
#         traj_original, interval_value, time = tr.exp_trajectory_quat(file,num_nodes)
#         traj = tr.exp_trajectory_quat_2_myobj(traj_original)

#         state_symbols = tuple(q+w+u0)
#         num_states = len(state_symbols)
#         specified_symbols = tuple(activations)
#         num_inputs = len(specified_symbols)
#         t = me.dynamicsymbols._t
#         objective_traj,objective_traj_jac = eq.custom_objective_quat(len(q),interval_value)

#         def obj(free):
#             # min_traj = traj_w * interval_value * np.sum((traj - free[:13*num_nodes])**2)
#             min_traj = traj_w * np.sum(objective_traj(np.split(free[:13*num_nodes],13),traj))
#             min_vel = vel_w * interval_value * np.sum((free[13*num_nodes:num_states*num_nodes])**2)
#             min_torque = act_w * interval_value * np.sum(free[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
#             return min_traj + min_torque + min_vel

#         def obj_grad(free):
#             grad = np.zeros_like(free)
#             # grad[:13*num_nodes] = traj_w * 2.0 * interval_value * (free[:13*num_nodes] - traj)
#             grad[:13*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:13*num_nodes],13),traj))
#             grad[13*num_nodes:num_states*num_nodes] = vel_w * 2 * interval_value * free[13*num_nodes:num_states*num_nodes]
#             grad[num_states*num_nodes:(num_states + num_inputs)*num_nodes] = act_w * 2.0 * interval_value * free[num_states*num_nodes:(num_states + num_inputs)*num_nodes]
#             return grad
#         instance_constraints = []
#         # for i in range(13):
#         instance_constraints.append(state_symbols[-3].func(0.0)-0) 
            
#         bounds1 = (0.0,1.0)
#         bounds = (bounds1,)*len(activations)
#         bndrs = dict(zip(activations,bounds))
#         start = tm.time()
#         prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
#                     num_nodes, interval_value,
#                     known_parameter_map={},
#                     instance_constraints=instance_constraints,
#                     bounds=bndrs,
#                     integration_method='midpoint')


#         time_to_create = tm.time() - start
#         print(time_to_create)



#         prob.add_option('max_iter',3000)
#         prob.add_option('limited_memory_max_history', 40)
#         initial_guess = np.zeros(prob.num_free)
#         initial_guess[:13*num_nodes] = traj_original.flatten()
#         time_2_solve_start = tm.time()
#         solution, info = prob.solve(initial_guess)
#         time_2_solve = tm.time() - time_2_solve_start
#         print(info['status_msg'])
#         print(info['obj_val'])
#         act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
#         print('Objective activations: ', act_obj)
#         reload (tr);
#         import matplotlib.pyplot as plt
#         tr.plot_results(solution,traj,time,num_nodes)
#         # import matplotlib.pyplot as plt
#         # fig, axes = plt.subplots(int(num_states+num_inputs), 1, sharex=True,
#         #                          figsize=(6.4, 0.8*(num_states+num_inputs)),
#         #                          layout='compressed')
#         # prob.plot_trajectories(solution, axes=axes)
#         import matplotlib.pyplot as plt
#         fig, axs = plt.subplots(13)
#         for j in range(13):
#             axs[j].plot(time,traj[j,:])
#             axs[j].plot(time,solution[j*num_nodes:(j+1)*num_nodes])
#             fig.set_figheight(10)
#         import trajectory_lib as tr
#         reload (tr);
#         # num_iter_sol = int(input('enter number of iterarions:'))
#         file_name = '../Motions/'+motion_folder+'/' + struct_name + '.mat'
#         tr.sol2struct(solution,activations,len(q),num_states,num_nodes,time,0,time_2_solve,file_name)

#         file_name_mot = '../Motions/'+motion_folder+'/' + struct_name + '.mot'
#         tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot)
